# Лабораторна робота №1: Первинне дослідження джерел даних (CityFlow S08)
**Студент:** Tim
**Середовище:** Arch Linux, Neovim, uv, Python 3.12+

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Визначення кореня проєкту
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

raw_path = root / "data" / "raw"
sample_path = root / "data" / "sample"
docs_path = root / "docs"
sample_path.mkdir(parents=True, exist_ok=True)
docs_path.mkdir(parents=True, exist_ok=True)

# 1. Зчитування сирих CSV
customers = pd.read_csv(raw_path / "customers.csv")
merchants = pd.read_csv(raw_path / "merchants.csv")
products = pd.read_csv(raw_path / "products.csv")
orders = pd.read_csv(raw_path / "orders.csv")

dfs = {
    "customers": (customers, "customer_id"),
    "merchants": (merchants, "merchant_id"),
    "products": (products, "product_id"),
    "orders": (orders, "order_id")
}

## 1. Збереження вибірки (Sample Data)
Збереження перших 100 рядків кожного джерела для відтворюваності в Git без raw-даних.

In [2]:
for name, (df, _) in dfs.items():
    df.head(100).to_csv(sample_path / f"{name}_sample.csv", index=False)
print("Семпли збережено в data/sample/")

Семпли збережено в data/sample/


## 2. Базові характеристики та Grain
- **customers**: Grain — один зареєстрований користувач; ключ — customer_id
- **merchants**: Grain — один заклад-партнер; ключ — merchant_id
- **products**: Grain — одна позиція меню/товару; ключ — product_id; FK: merchant_id
- **orders**: Grain — одна транзакція замовлення; ключ — order_id; FK: customer_id, merchant_id

In [3]:
for name, (df, key) in dfs.items():
    print(f"=== {name.upper()} ===")
    print(f"Розмірність (rows, cols): {df.shape}")
    print(f"Кандидатний ключ: {key}")
    print(f"Типи даних:\n{df.dtypes}\n")

=== CUSTOMERS ===
Розмірність (rows, cols): (801, 8)
Кандидатний ключ: customer_id
Типи даних:
customer_id            str
full_name              str
email                  str
phone              float64
city                   str
service_zone_id        str
registered_at          str
customer_status        str
dtype: object

=== MERCHANTS ===
Розмірність (rows, cols): (43, 7)
Кандидатний ключ: merchant_id
Типи даних:
merchant_id        str
merchant_name      str
merchant_type      str
city               str
service_zone_id    str
opened_at          str
merchant_status    str
dtype: object

=== PRODUCTS ===
Розмірність (rows, cols): (534, 7)
Кандидатний ключ: product_id
Типи даних:
product_id            str
merchant_id           str
product_name          str
category              str
unit_price_uah    float64
is_active            bool
updated_at            str
dtype: object

=== ORDERS ===
Розмірність (rows, cols): (6001, 10)
Кандидатний ключ: order_id
Типи даних:
order_id               

## 3. Перевірка пропусків і порожніх значень
Враховуються: системні null (NaN/None), порожні рядки ('') та пробіли ('   ').

In [4]:
def audit_nulls(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for col in df.columns:
        null_count = df[col].isna().sum()
        empty_str_count = 0
        whitespace_count = 0
        if df[col].dtype == "object":
            empty_str_count = (df[col] == "").sum()
            whitespace_count = df[col].astype(str).str.strip().eq("").sum() - null_count - empty_str_count
            if whitespace_count < 0:
                whitespace_count = 0
        total_missing = null_count + empty_str_count + whitespace_count
        records.append({
            "column": col,
            "nulls": null_count,
            "empty_strings": empty_str_count,
            "whitespace_only": whitespace_count,
            "total_missing": total_missing,
            "missing_pct": round(total_missing / len(df) * 100, 2)
        })
    return pd.DataFrame(records)

for name, (df, _) in dfs.items():
    print(f"--- Пропуски у {name} ---")
    missing_report = audit_nulls(df)
    print(missing_report[missing_report["total_missing"] > 0])
    print()

--- Пропуски у customers ---
  column  nulls  empty_strings  whitespace_only  total_missing  missing_pct
3  phone      5              0                0              5         0.62

--- Пропуски у merchants ---
Empty DataFrame
Columns: [column, nulls, empty_strings, whitespace_only, total_missing, missing_pct]
Index: []

--- Пропуски у products ---
Empty DataFrame
Columns: [column, nulls, empty_strings, whitespace_only, total_missing, missing_pct]
Index: []

--- Пропуски у orders ---
Empty DataFrame
Columns: [column, nulls, empty_strings, whitespace_only, total_missing, missing_pct]
Index: []



## 4. Перевірка дублікатів (повні рядки та кандидатні ключі)

In [5]:
for name, (df, key) in dfs.items():
    full_dups = df.duplicated().sum()
    key_dups = df.duplicated(subset=[key]).sum()
    print(f"[{name}] Повні дублікати: {full_dups} | Дублікати ключа '{key}': {key_dups}")
    if key_dups > 0:
        print(f"Приклади колізій ключа в {name}:")
        print(df[df.duplicated(subset=[key], keep=False)].head(4))
        print()

[customers] Повні дублікати: 1 | Дублікати ключа 'customer_id': 1
Приклади колізій ключа в customers:
     customer_id    full_name                            email         phone  \
503  S08-C000504  Olena Koval  s08.customer000504@example.test  3.809382e+11   
800  S08-C000504  Olena Koval  s08.customer000504@example.test  3.809382e+11   

     city service_zone_id                   registered_at customer_status  
503  Lviv         LVI-Z02  2026-03-09T23:37:51.562482847Z          active  
800  Lviv         LVI-Z02  2026-03-09T23:37:51.562482847Z          active  

[merchants] Повні дублікати: 1 | Дублікати ключа 'merchant_id': 1
Приклади колізій ключа в merchants:
   merchant_id     merchant_name merchant_type     city service_zone_id  \
8   S08-M00009  Market Market 09       grocery  Kharkiv         KHA-Z03   
42  S08-M00009  Market Market 09       grocery  Kharkiv         KHA-Z03   

                         opened_at merchant_status  
8   2021-05-23T16:08:07.469483627Z          act

## 5. Перевірка зв'язків (Foreign Keys цілісність)

In [6]:
# Перевірка: orders.customer_id -> customers.customer_id
orphan_orders_cust = orders[~orders["customer_id"].isin(customers["customer_id"])]
print(f"Замовлень із неіснуючим customer_id: {len(orphan_orders_cust)}")

# Перевірка: orders.merchant_id -> merchants.merchant_id
orphan_orders_merch = orders[~orders["merchant_id"].isin(merchants["merchant_id"])]
print(f"Замовлень із неіснуючим merchant_id: {len(orphan_orders_merch)}")

# Перевірка: products.merchant_id -> merchants.merchant_id
orphan_prods_merch = products[~products["merchant_id"].isin(merchants["merchant_id"])]
print(f"Товарів із неіснуючим merchant_id: {len(orphan_prods_merch)}")

Замовлень із неіснуючим customer_id: 1
Замовлень із неіснуючим merchant_id: 1
Товарів із неіснуючим merchant_id: 1


## 6. Статистичний огляд колонок

In [7]:
print("--- Статистика Orders (числові поля) ---")
print(orders.describe())

print("\n--- Категоріальні значення Orders (статуси) ---")
print(orders["order_status"].value_counts(dropna=False))

--- Статистика Orders (числові поля) ---
       subtotal_amount  delivery_fee  discount_amount  total_amount
count      6001.000000   6001.000000      6001.000000   6001.000000
mean        547.871698     72.954254        17.187265    603.661548
std         432.973720     28.494591        39.662595    409.483281
min           1.400000      0.000000         0.000000      3.260000
25%         214.430000     69.630000         0.000000    289.180000
50%         432.940000     79.120000         0.000000    503.300000
75%         768.330000     89.100000        13.430000    826.170000
max        3033.610000    148.880000       350.000000   3102.010000

--- Категоріальні значення Orders (статуси) ---
order_status
delivered             4966
cancelled              449
partially_refunded     223
refunded               198
failed                 162
in_progress              3
Name: count, dtype: int64


## 7. Підсумкова діагностична таблиця

In [8]:
summary_records = []
for name, (df, key) in dfs.items():
    min_date, max_date = "N/A", "N/A"
    for date_col in ["created_at", "registered_at", "opened_at"]:
        if date_col in df.columns:
            temp_dates = pd.to_datetime(df[date_col], errors="coerce")
            min_date = str(temp_dates.min())
            max_date = str(temp_dates.max())
            break
            
    summary_records.append({
        "dataset": name,
        "rows_count": len(df),
        "columns_count": df.shape[1],
        "unique_keys": df[key].nunique(),
        "key_duplicates": df.duplicated(subset=[key]).sum(),
        "full_row_duplicates": df.duplicated().sum(),
        "total_null_cells": df.isna().sum().sum(),
        "min_date": min_date,
        "max_date": max_date
    })

summary_df = pd.DataFrame(summary_records)
print(summary_df.to_string(index=False))

  dataset  rows_count  columns_count  unique_keys  key_duplicates  full_row_duplicates  total_null_cells                            min_date                            max_date
customers         801              8          800               1                    1                 5 2024-03-10 04:22:08.504394413+00:00 2026-08-29 07:06:33.137727612+00:00
merchants          43              7           42               1                    1                 0 2019-09-07 11:46:11.601515588+00:00 2025-07-19 12:35:09.191823059+00:00
 products         534              7          533               1                    1                 0                                 N/A                                 N/A
   orders        6001             10         6000               1                    1                 0           2025-09-01 11:47:50+00:00           2026-08-31 20:31:25+00:00


## 8. Генерація діагностичних графіків
Графіки зберігаються у docs/ як PNG без використання GUI.

In [9]:
# Графік 1: Кількість замовлень за статусами
plt.figure(figsize=(8, 4.5))
status_counts = orders["order_status"].fillna("NULL/MISSING").value_counts()
status_counts.plot(kind="bar", color="#2b5c8f", edgecolor="black")
plt.title("Розподіл замовлень за статусами (CityFlow S08)")
plt.xlabel("Статус замовлення")
plt.ylabel("Кількість записів")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(docs_path / "orders_by_status.png", dpi=300)
plt.close()

# Графік 2: Динаміка замовлень за датою
orders["order_date"] = pd.to_datetime(orders["created_at"], errors="coerce").dt.date
date_counts = orders["order_date"].value_counts().sort_index()

plt.figure(figsize=(10, 4.5))
date_counts.plot(kind="line", marker="o", color="#d95f02", linewidth=1.5)
plt.title("Кількість замовлень за датами (CityFlow S08)")
plt.xlabel("Дата")
plt.ylabel("Кількість замовлень")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(docs_path / "orders_by_date.png", dpi=300)
plt.close()

print("Графіки збережено в docs/: orders_by_status.png, orders_by_date.png")

Графіки збережено в docs/: orders_by_status.png, orders_by_date.png
